In [1]:
import sys
from pathlib import Path

# this allows the importing of modules from ../modules
root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from modules import ImagenetLoader

loader = ImagenetLoader()

c:\Users\fiach\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_50 = loader.load_radimagenet_resnet50(
    weights_path="trained_models/resnet50_baseline_gpu_new/resnet50_baseline_gpu_new.pth", 
    load_type="load" 
)

model_18 = loader.load_radimagenet_resnet18(
    weights_path="trained_models/resnet18_baseline_gpu_new/resnet18_baseline_gpu_new.pth",
    load_type="load"
    )

model_10t = loader.load_radimagenet_resnet10t(
    weights_path="trained_models/resnet10t_baseline_gpu_new/resnet10t_baseline_gpu_new.pth",
    load_type="load"
)

In [3]:
import os
import torch

def get_model_size(model, label="Model"):
    torch.save(model.state_dict(), "temp.p")
    size_mb = os.path.getsize("temp.p") / (1024 * 1024)
    os.remove("temp.p")
    print(f"{label} size: {size_mb:.2f} MB")
    return size_mb

In [4]:
from modules import datasetPrepper, ModelEvaluator

data = datasetPrepper(
    "data/labels.csv",
    image_dir="data/test_images"
    ).prepare()

evaluator = ModelEvaluator(
    data_loader=data.val_loader,
    class_names=data.class_names
)

In [5]:
print(next(model_50.parameters()).dtype)
print(next(model_18.parameters()).dtype)
print(next(model_10t.parameters()).dtype)

torch.float32
torch.float32
torch.float32


In [ ]:
import torch
import copy
from torchao.quantization import (
    quantize_, 
    Int8DynamicActivationInt8WeightConfig,
    Int8WeightOnlyConfig,
    Int4WeightOnlyConfig,
    Float8WeightOnlyConfig
)

def run_quantization_suite(model_label, base_model, evaluator):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"--- Running on: {device.type.upper()} ---")
    base_model = base_model.to(device).to(torch.float32)
    
    techniques = {
        "INT8_Dyn": Int8DynamicActivationInt8WeightConfig(),
        "INT8_WO": Int8WeightOnlyConfig(),
        "INT4_WO": Int4WeightOnlyConfig(group_size=128),
    }
    
    if device.type == "cuda":
        techniques["FP8_WO"] = Float8WeightOnlyConfig()
        techniques["FP16"] = "fp16_cast" 
    else:
        techniques["BF16"] = "bf16_cast"

    models_to_test = {f"{model_label}_Baseline_FP32": base_model}
    
    for suffix, config in techniques.items():
        print(f"Preparing {model_label}_{suffix}...")
        test_model = copy.deepcopy(base_model)
        
        try:
            if config == "fp16_cast":
                test_model = test_model.to(torch.float16)
            elif config == "bf16_cast":
                test_model = test_model.to(torch.bfloat16)
            else:
                quantize_(test_model, config)
            
            test_model = torch.compile(test_model)
            
            models_to_test[f"{model_label}_{suffix}"] = test_model
            
        except Exception as e:
            print(f"   [Error] {suffix} failed: {e}")

    print(f"Sending {len(models_to_test)} models to the evaluator...")
    evaluator.evaluate_many(models_to_test)
    
    return evaluator

In [10]:
eval = run_quantization_suite("ResNet 50", model_50, evaluator)

--- Running on: CPU ---
Preparing ResNet 50_INT8_Dyn...
Preparing ResNet 50_INT8_WO...
Preparing ResNet 50_INT4_WO...
   [Error] INT4_WO failed: Requires mslk >= 1.0.0
Preparing ResNet 50_BF16...
Sending 4 models to the evaluator...

[Evaluating] ResNet 50_Baseline_FP32...

Warming up ResNet 50_Baseline_FP32...


c:\Users\fiach\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchao\quantization\quant_api.py:925: UserWarning: Config Deprecation: version 1 of Int8WeightOnlyConfig is deprecated and will no longer be supported in a future release, please use version 2, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(


Running inference...

[Evaluating] ResNet 50_INT8_Dyn...

Warming up ResNet 50_INT8_Dyn...


InductorError: RuntimeError: Compiler: cl is not found.

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


In [ ]:
evaluator_18 = ModelEvaluator(
    data_loader=data.val_loader,
    class_names=data.class_names
)

eval_18 = run_quantization_suite("ResNet 18", model_18, evaluator_18)

In [ ]:
evaluator_10t = ModelEvaluator(
    data_loader=data.val_loader,
    class_names=data.class_names
)

eval_10t = run_quantization_suite("ResNet 10t", model_18, evaluator_10t)